In [1]:
import pandas as pd
import numpy as np
import torch


with open('./dataset.txt','r',encoding = 'utf-8') as f:
    text = f.read()

print(len(text))

1115394


In [2]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [3]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

'"hen'

In [4]:
data = torch.tensor(encode(text), dtype=torch.long)
data.to('cuda')
print(data.shape,data.dtype)
print(data[:1000])

torch.Size([338025]) torch.int64
tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597,  2252,    11,
         3285,   502,  2740,    13,   198,   198,  3237,    25,   198,  5248,
          461,    11,  2740,    13,   198,   198,  5962, 22307,    25,   198,
         1639,   389,   477, 12939,  2138,   284,  4656,   621,   284,  1145,
          680,    30,   198,   198,  3237,    25,   198,  4965,  5634,    13,
        12939,    13,   198,   198,  5962, 22307,    25,   198,  5962,    11,
          345,   760,   327,  1872,   385,  1526, 28599,   318,  4039,  4472,
          284,   262,   661,    13,   198,   198,  3237,    25,   198,  1135,
          760,   470,    11,   356,   760,   470,    13,   198,   198,  5962,
        22307,    25,   198,  5756,   514,  1494,   683,    11,   290,   356,
         1183,   423, 11676,   379,   674,   898,  2756,    13,   198,  3792,
          470,   257, 15593,    30,   198,   198,  3237,    25,   198,  2949,
          517,  3375,   319,   

In [6]:
n = int(0.9*len(data)) # 训练集和验证集
train_data = data[:n]
val_data = data[n:]


In [7]:
# train chunks
block_size = 8
# block size = 16
train_data[:block_size+1]

tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597,  2252])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'{decode(context.tolist())},   {decode([target.tolist()])}')

First,    Citizen
First Citizen,   :
First Citizen:,   

First Citizen:
,   Before
First Citizen:
Before,    we
First Citizen:
Before we,    proceed
First Citizen:
Before we proceed,    any
First Citizen:
Before we proceed any,    further


In [11]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    '''
    generate a small batch of input and out put
    '''
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

xb,yb = get_batch('train')

print(xb)
print(xb.shape)

print('---------')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f'当输入是{context}的时候，目标是  {target}')

tensor([[  198, 30313,   262, 22397,   282,   290,   884,  3790],
        [ 4151,   438,   198, 10418,   329,   511, 11989,    11],
        [ 3355,   322, 12105,   287,  3426,  6729,   198,  3886],
        [  290, 15581,  8636,    13,   198,   198, 35510,  4221]])
torch.Size([4, 8])
---------
当输入是tensor([198])的时候，目标是  30313
当输入是tensor([  198, 30313])的时候，目标是  262
当输入是tensor([  198, 30313,   262])的时候，目标是  22397
当输入是tensor([  198, 30313,   262, 22397])的时候，目标是  282
当输入是tensor([  198, 30313,   262, 22397,   282])的时候，目标是  290
当输入是tensor([  198, 30313,   262, 22397,   282,   290])的时候，目标是  884
当输入是tensor([  198, 30313,   262, 22397,   282,   290,   884])的时候，目标是  3790
当输入是tensor([  198, 30313,   262, 22397,   282,   290,   884,  3790])的时候，目标是  198
当输入是tensor([4151])的时候，目标是  438
当输入是tensor([4151,  438])的时候，目标是  198
当输入是tensor([4151,  438,  198])的时候，目标是  10418
当输入是tensor([ 4151,   438,   198, 10418])的时候，目标是  329
当输入是tensor([ 4151,   438,   198, 10418,   329])的时候，目标是  511
当输入是tensor([ 4151,   438,

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        # 每个 Token 对应 vocab_size 个预测分数
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            vocab_size
        )

    def forward(self, idx, targets=None):
        # idx.shape = (B, T)
        logits = self.token_embedding_table(idx)

        # logits.shape = (B, T, C)
        # C = vocab_size
        return logits

m = BigramLanguageModel(vocab_size)

out = m(xb, yb)
print(out.shape)

IndexError: index out of range in self